In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
import torch

assert torch.cuda.is_available(), 'Hãy bật Accelerator GPU trong Kaggle Settings'
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

GPU count: 2
0 Tesla T4
1 Tesla T4


In [3]:
def locate_project():
    candidates = sorted({path.parent for path in Path('/kaggle/input').rglob('train_mid.py')
                         if (path.parent / 'data' / 'split_manifest.csv').is_file()})
    if not candidates:
        archives = list(Path('/kaggle/input').rglob('TickNets_mid224_split_100_100.zip'))
        if len(archives) == 1:
            destination = Path('/kaggle/working/source_bundle')
            shutil.unpack_archive(archives[0], destination)
            candidates = sorted({path.parent for path in destination.rglob('train_mid.py')
                                 if (path.parent / 'data' / 'split_manifest.csv').is_file()})
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one TickNets project, found: {candidates}')
    return candidates[0]

PROJECT_ROOT = locate_project()
DATA_ROOT = PROJECT_ROOT / 'data'
OUTPUT_DIR = Path('/kaggle/working/runs/c_mid224_seed42')
print('Project:', PROJECT_ROOT)
print('Data:', DATA_ROOT)
print('Output:', OUTPUT_DIR)

Project: /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets
Data: /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/data
Output: /kaggle/working/runs/c_mid224_seed42


In [4]:
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Output already exists: {OUTPUT_DIR}')
command = [
    sys.executable, str(PROJECT_ROOT / 'train_mid.py'),
    '--data-root', str(DATA_ROOT), '--variant', 'Mid224', '--model', 'c',
    '--seed', '42', '--epochs', '200', '--stop-after-epoch', '100',
    '--batch-size', '64', '--num-workers', '0', '--device', 'auto',
    '--output-dir', str(OUTPUT_DIR),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

Running: /usr/bin/python3 /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/train_mid.py --data-root /kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/data --variant Mid224 --model c --seed 42 --epochs 200 --stop-after-epoch 100 --batch-size 64 --num-workers 0 --device auto --output-dir /kaggle/working/runs/c_mid224_seed42
{"epoch": 1, "learning_rate": 0.1, "train_loss": 1.0206017433547974, "train_top1": 56.976, "train_samples": 25000}
{"epoch": 2, "learning_rate": 0.09999383162408304, "train_loss": 0.7007736782455445, "train_top1": 68.98, "train_samples": 25000}
{"epoch": 3, "learning_rate": 0.09997532801828658, "train_loss": 0.6163661300468445, "train_top1": 73.444, "train_samples": 25000}
{"epoch": 4, "learning_rate": 0.09994449374809851, "train_loss": 0.5665150390243531, "train_top1": 75.868, "train_samples": 25000}
{"epoch": 5, "learning_rate": 0.09990133642141359, "train_loss": 0.5113299475479126, "train_top1": 78.796, "train_

CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/train_mid.py', '--data-root', '/kaggle/input/datasets/lastapple/dl-btgk-modelc-mid224-epoch001-100/TickNets/data', '--variant', 'Mid224', '--model', 'c', '--seed', '42', '--epochs', '200', '--stop-after-epoch', '100', '--batch-size', '64', '--num-workers', '0', '--device', 'auto', '--output-dir', '/kaggle/working/runs/c_mid224_seed42'], returncode=0)

In [5]:
checkpoint = torch.load(OUTPUT_DIR / 'last.pt', map_location='cpu', weights_only=True)
assert checkpoint['epoch'] == 100
assert checkpoint['config']['model'] == 'c'
assert checkpoint['config']['variant'] == 'Mid224'
assert 'rng_state' in checkpoint
assert not (OUTPUT_DIR / 'test_metrics.json').exists()
with (OUTPUT_DIR / 'epochs.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 100 and int(rows[-1]['epoch']) == 100
archive = shutil.make_archive('/kaggle/working/c_mid224_seed42_epoch100', 'zip',
                               root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print(json.dumps({'status': 'ready_for_session_2', 'epoch': 100, 'run': str(OUTPUT_DIR),
                  'download_archive': archive}, indent=2))

{
  "status": "ready_for_session_2",
  "epoch": 100,
  "run": "/kaggle/working/runs/c_mid224_seed42",
  "download_archive": "/kaggle/working/c_mid224_seed42_epoch100.zip"
}
